In [6]:
!pip3 install --upgrade --force-reinstall --no-cache-dir --break-system-packages \
  "llama-index-core==0.14.15" \
  "llama-index-llms-huggingface-api==0.6.2" \
  "llama-index-embeddings-huggingface==0.6.1" \
  "llama-index-retrievers-bm25==0.5.2" \
  "sentence-transformers==5.1.0" \
  "transformers>=4.46,<5.0" \
  "huggingface_hub>=0.24,<1.0" \
  "rank-bm25==0.2.2" \
  "PyStemmer==2.2.0.3"


INFO: pip is looking at multiple versions of llama-index-retrievers-bm25 to determine which version is compatible with other requirements. This could take a while.
ERROR: Cannot install llama-index-core==0.14.15, llama-index-embeddings-huggingface==0.6.1, llama-index-llms-huggingface-api==0.6.2 and llama-index-retrievers-bm25==0.5.2 because these package versions have conflicting dependencies.

The conflict is caused by:
    The user requested llama-index-core==0.14.15
    llama-index-llms-huggingface-api 0.6.2 depends on llama-index-core<0.15 and >=0.13.0
    llama-index-embeddings-huggingface 0.6.1 depends on llama-index-core<0.15 and >=0.13.0
    llama-index-retrievers-bm25 0.5.2 depends on llama-index-core<0.13.0 and >=0.12.0

To fix this you could try to:
1. loosen the range of package versions you've specified
2. remove package versions to allow pip to attempt to solve the dependency conflict

ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/depend

In [15]:
import os
import json
from typing import List, Optional
import asyncio
import warnings
import numpy as np
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()


# Core LlamaIndex imports
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    Document,
    Settings,
    DocumentSummaryIndex,
    KeywordTableIndex,
)
from llama_index.core.retrievers import (
    BaseRetriever,
    VectorIndexRetriever,
    AutoMergingRetriever,
    RecursiveRetriever,
    QueryFusionRetriever,
)
from llama_index.core.indices.document_summary import (
    DocumentSummaryIndexLLMRetriever,
    DocumentSummaryIndexEmbeddingRetriever,
)
from llama_index.core.node_parser import SentenceSplitter, HierarchicalNodeParser
from llama_index.core.schema import NodeWithScore, QueryBundle
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.embeddings import BaseEmbedding

# Hugging Face LlamaIndex integration
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI

# Advanced retriever imports
from llama_index.retrievers.bm25 import BM25Retriever

# Sentence transformers
from sentence_transformers import SentenceTransformer

# Statistical libraries for fusion techniques
try:
    from scipy import stats
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False
    print("⚠️ scipy not available - some advanced fusion features will be limited")

print("✅ All imports successful!")

✅ All imports successful!


In [13]:
# Hugging Face Mistral LLM using official LlamaIndex HF integration
def create_huggingface_llm():
    """Create Hugging Face Mistral LLM instance using LlamaIndex integration."""
    try:
        token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
        if not token:
            raise ValueError("Set HF_TOKEN (or HUGGINGFACEHUB_API_TOKEN).")

        llm = HuggingFaceInferenceAPI(
            model_name="mistralai/Mistral-Small-3.1-24B-Instruct-2503",
            token=token,
            task="conversational",
            temperature=0.9,
            max_new_tokens=512,
        )
        print("✅ Hugging Face Mistral LLM initialized")
        return llm

    except Exception as e:
        print(f"⚠️ Hugging Face initialization error: {e}")
        print("Falling back to mock LLM for demonstration")

        from llama_index.core.llms.mock import MockLLM
        return MockLLM(max_tokens=512)


In [16]:
# Initialize embedding model first
print("🔧 Initializing HuggingFace embeddings...")
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)
print("✅ HuggingFace embeddings initialized!")

# Setup with watsonx.ai
print("🔧 Initializing watsonx.ai LLM...")
llm = create_huggingface_llm()

# Configure global settings
Settings.llm = llm
Settings.embed_model = embed_model
print("✅ HuggingFace LLM and embeddings configured!")

🔧 Initializing HuggingFace embeddings...
✅ HuggingFace embeddings initialized!
🔧 Initializing watsonx.ai LLM...
✅ Hugging Face Mistral LLM initialized
✅ HuggingFace LLM and embeddings configured!


### Advanced Retrievers

Advanced retrievers in LlamaIndex go beyond basic vector search by combining semantic search, keyword matching, hierarchical context, multi-query retrieval, and result-fusion methods for smarter, more context-aware retrieval.

##### Advantages

1. Improved Accuracy: Uses multiple strategies to find more relevant results.

2. Better Context Preservation: Keeps key relationships between information pieces.

3. Reduced Hallucination: More precise retrieval improves response correctness.

4. Scalability: Works efficiently on large document collections.

5. Flexibility: Combines methods to optimize retrieval quality.

##### Index Types Overview

1. `VectorStoreIndex`: Stores chunk embeddings for semantic retrieval and is widely used in RAG/LLM pipelines.

2. `DocumentSummaryIndex`: Stores document summaries to pre-filter retrieval, making it useful for large corpora that exceed LLM context limits.

3. `KeywordTableIndex`: Maps keywords to chunks for exact term matching in rule-based or hybrid retrieval.

In [17]:
# Sample data for the lab - AI/ML focused documents
SAMPLE_DOCUMENTS = [
    "Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.",
    "Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.",
    "Natural language processing enables computers to understand, interpret, and generate human language.",
    "Computer vision allows machines to interpret and understand visual information from the world.",
    "Reinforcement learning is a type of machine learning where agents learn to make decisions through rewards and penalties.",
    "Supervised learning uses labeled training data to learn a mapping from inputs to outputs.",
    "Unsupervised learning finds hidden patterns in data without labeled examples.",
    "Transfer learning leverages knowledge from pre-trained models to improve performance on new tasks.",
    "Generative AI can create new content including text, images, code, and more.",
    "Large language models are trained on vast amounts of text data to understand and generate human-like text."
]

# Consistent query examples used throughout the lab
DEMO_QUERIES = {
    "basic": "What is machine learning?",
    "technical": "neural networks deep learning", 
    "learning_types": "different types of learning",
    "advanced": "How do neural networks work in deep learning?",
    "applications": "What are the applications of AI?",
    "comprehensive": "What are the main approaches to machine learning?",
    "specific": "supervised learning techniques"
}

print(f"📄 Loaded {len(SAMPLE_DOCUMENTS)} sample documents")
print(f"🔍 Prepared {len(DEMO_QUERIES)} consistent demo queries")
for i, doc in enumerate(SAMPLE_DOCUMENTS[:3], 1):
    print(f"{i}. {doc}")
print("...")

📄 Loaded 10 sample documents
🔍 Prepared 7 consistent demo queries
1. Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.
2. Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.
3. Natural language processing enables computers to understand, interpret, and generate human language.
...


In [19]:
class AdvancedRetrieversLab:
    def __init__(self):
        print("🚀 Initializing Advanced Retrievers Lab...")
        self.documents = [Document(text=text) for text in SAMPLE_DOCUMENTS]
        self.nodes = SentenceSplitter().get_nodes_from_documents(self.documents)
        
        print("📊 Creating indexes...")
        # Create various indexes
        self.vector_index = VectorStoreIndex.from_documents(self.documents)
        self.document_summary_index = DocumentSummaryIndex.from_documents(self.documents)
        self.keyword_index = KeywordTableIndex.from_documents(self.documents)
        
        print("✅ Advanced Retrievers Lab Initialized!")
        print(f"📄 Loaded {len(self.documents)} documents")
        print(f"🔢 Created {len(self.nodes)} nodes")

# Initialize the lab
lab = AdvancedRetrieversLab()

🚀 Initializing Advanced Retrievers Lab...
📊 Creating indexes...
current doc id: 0498f11a-799a-4467-a0e2-e6fa9c187b8e
current doc id: 77db821a-40df-4b1d-a196-7e0246468bfd
current doc id: 7306ee5f-a9be-4cbd-98f1-5bf708da71f3
current doc id: 12fa5991-9eff-4794-a7a0-2a3949d79fca
current doc id: 22d2bdb4-d36a-4d2e-aadb-bdc54a65d915
current doc id: b0f38768-ec9c-448b-8b01-f9ef9aab9b0c
current doc id: 057f955c-fc2d-4f66-9ac2-3b8acdd3d80c
current doc id: c5f94bf3-f7d8-41c9-93f5-3245fead3b91
current doc id: 874fb191-2451-4dbd-8372-5714bbb8b87e
current doc id: 1271dc6c-7632-490a-90ac-c6a9bd4f09fc
✅ Advanced Retrievers Lab Initialized!
📄 Loaded 10 documents
🔢 Created 10 nodes


### Vector Index Retriever (The Foundation)

- **How it works:** It splits documents into nodes, embeds both nodes and query in the same vector space, and returns top nodes ranked by cosine similarity (typically batched at 2048 nodes).

- **When to use:** Use it for general semantic search and RAG scenarios where meaning matters more than exact keyword matches.  

- **Key characteristics:** It stores per-chunk embeddings in `VectorStoreIndex` and is widely used for context-aware retrieval in LLM pipelines.  

- **Strengths:** It captures semantic meaning well, handles synonyms and related concepts, and works naturally with free-text queries.  

- **Limitations:** It may miss exact term matches, depends on embedding quality, and can be expensive on very large datasets.

In [20]:
print("=" * 60)
print("1. VECTOR INDEX RETRIEVER")
print("=" * 60)

# Basic vector retriever
vector_retriever = VectorIndexRetriever(
    index=lab.vector_index,
    similarity_top_k=3
)

# Alternative creation method
alt_retriever = lab.vector_index.as_retriever(similarity_top_k=3)

query = DEMO_QUERIES["basic"]  # "What is machine learning?"
nodes = vector_retriever.retrieve(query)

print(f"Query: {query}")
print(f"Retrieved {len(nodes)} nodes:")
for i, node in enumerate(nodes, 1):
    print(f"{i}. Score: {node.score:.4f}")
    print(f"   Text: {node.text[:100]}...")
    print()

1. VECTOR INDEX RETRIEVER
Query: What is machine learning?
Retrieved 3 nodes:
1. Score: 0.8700
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Score: 0.7644
   Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re...

3. Score: 0.6979
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs....



### BM25 Retriever (Advanced Keyword-Based Search)

BM25 is a production-grade keyword retriever (used in Elasticsearch/Lucene) that improves TF-IDF for better precision and ranking.

- **TF-IDF basics**: TF measures term frequency in a document, IDF measures term rarity across documents, and TF-IDF highlights terms that are frequent in one doc but rare globally.

- **BM25 improvements**: It adds term-frequency saturation, document-length normalization, and tunable parameters (k1, b) for more stable scoring.

- **When to use**: Best for exact-term retrieval in technical, legal, catalog, or academic content where wording precision matters.

- **Advantages**: It is fast, interpretable, training-free, and highly effective for keyword matching in production systems.

- **Limitations**: It lacks semantic understanding, handles synonyms/typos poorly, has limited context awareness, and needs parameter tuning.


In [21]:
print("=" * 60)
print("2. BM25 RETRIEVER")
print("=" * 60)

try:
    import Stemmer
    
    # Create BM25 retriever with default parameters
    bm25_retriever = BM25Retriever.from_defaults(
        nodes=lab.nodes,
        similarity_top_k=3,
        stemmer=Stemmer.Stemmer("english"),
        language="english"
    )
    
    query = DEMO_QUERIES["technical"]  # "neural networks deep learning"
    nodes = bm25_retriever.retrieve(query)
    
    print(f"Query: {query}")
    print("BM25 analyzes exact keyword matches with sophisticated scoring")
    print(f"Retrieved {len(nodes)} nodes:")
    
    for i, node in enumerate(nodes, 1):
        score = node.score if hasattr(node, 'score') and node.score else 0
        print(f"{i}. BM25 Score: {score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        
        # Highlight which query terms appear in the text
        text_lower = node.text.lower()
        query_terms = query.lower().split()
        found_terms = [term for term in query_terms if term in text_lower]
        if found_terms:
            print(f"   → Found terms: {found_terms}")
        print()
    
    print("BM25 vs TF-IDF Comparison:")
    print("TF-IDF Problem: Linear term frequency scaling")
    print("  Example: 10 occurrences → score of 10, 100 occurrences → score of 100")
    print("BM25 Solution: Saturation function")
    print("  Example: 10 occurrences → high score, 100 occurrences → slightly higher score")
    print()
    print("TF-IDF Problem: No document length consideration")
    print("  Example: Long documents dominate results")
    print("BM25 Solution: Length normalization (b parameter)")
    print("  Example: Scores adjusted based on document length vs. average")
    print()
    print("Key BM25 Parameters:")
    print("- k1 ≈ 1.2: Term frequency saturation (how quickly scores plateau)")
    print("- b ≈ 0.75: Document length normalization (0=none, 1=full)")
    print("- IDF weighting: Rare terms get higher scores")
        
except ImportError:
    print("⚠️ BM25Retriever requires 'pip install PyStemmer'")
    print("Demonstrating BM25 concepts with fallback vector search...")
    
    fallback_retriever = lab.vector_index.as_retriever(similarity_top_k=3)
    query = DEMO_QUERIES["technical"]
    nodes = fallback_retriever.retrieve(query)
    
    print(f"Query: {query}")
    print("(Using vector fallback to demonstrate BM25 concepts)")
    
    for i, node in enumerate(nodes, 1):
        print(f"{i}. Vector Score: {node.score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        
        # Demonstrate TF-IDF concept manually
        text_lower = node.text.lower()
        query_terms = query.lower().split()
        found_terms = [term for term in query_terms if term in text_lower]
        
        if found_terms:
            print(f"   → BM25 would boost this result for terms: {found_terms}")
        print()
    
    print("BM25 Concept Demonstration:")
    print("1. TF-IDF Foundation:")
    print("   - Term Frequency: How often words appear in document")
    print("   - Inverse Document Frequency: How rare words are across collection")
    print("   - TF-IDF = TF × IDF (balances frequency vs rarity)")
    print()
    print("2. BM25 Improvements:")
    print("   - Saturation: Prevents over-scoring repeated terms")
    print("   - Length normalization: Prevents long document bias")
    print("   - Tunable parameters: k1 (saturation) and b (length adjustment)")
    print()
    print("3. Real-world Usage:")
    print("   - Elasticsearch default scoring function")
    print("   - Apache Lucene/Solr standard")
    print("   - Used in 83% of text-based recommender systems")
    print("   - Developed by Robertson & Spärck Jones at City University London")

2. BM25 RETRIEVER
Query: neural networks deep learning
BM25 analyzes exact keyword matches with sophisticated scoring
Retrieved 3 nodes:
1. BM25 Score: 2.5203
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...
   → Found terms: ['neural', 'networks', 'deep', 'learning']

2. BM25 Score: 0.3372
   Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re...
   → Found terms: ['learning']

3. BM25 Score: 0.3024
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs....
   → Found terms: ['learning']

BM25 vs TF-IDF Comparison:
TF-IDF Problem: Linear term frequency scaling
  Example: 10 occurrences → score of 10, 100 occurrences → score of 100
BM25 Solution: Saturation function
  Example: 10 occurrences → high score, 100 occurrences → slightly higher score

TF-IDF Problem: No document length consideration
  Example: Long documents dominat

### Document Summary Index Retrievers

Document Summary Index Retrievers use stored summaries to filter candidates, then return the original full documents.

- **How it works:** Summaries are created at indexing time, used for first-pass filtering, and matched documents are returned in full.  

- **Retriever types:** `DocumentSummaryIndexLLMRetriever` uses an LLM for smarter but slower/costlier selection, while `DocumentSummaryIndexEmbeddingRetriever` uses summary embeddings for faster cheaper matching.  

- **When to use:** Best for large, diverse corpora where document-level filtering improves efficiency before deeper retrieval.  

- **Config:** Tune `choice_top_k` (LLM) or `similarity_top_k` (embedding), defaulting to 1.  

- **Strengths:** Efficiently narrows search space in heterogeneous collections while preserving full-document context.  

- **Limitations:** Needs summary generation at index time, may lose detail in summaries, and LLM-based retrieval can be slower and more expensive.

In [22]:
print("=" * 60)
print("3. DOCUMENT SUMMARY INDEX RETRIEVERS")
print("=" * 60)

# LLM-based document summary retriever
doc_summary_retriever_llm = DocumentSummaryIndexLLMRetriever(
    lab.document_summary_index,
    choice_top_k=3  # Number of documents to select
)

# Embedding-based document summary retriever  
doc_summary_retriever_embedding = DocumentSummaryIndexEmbeddingRetriever(
    lab.document_summary_index,
    similarity_top_k=3  # Number of documents to select
)

query = DEMO_QUERIES["learning_types"]  # "different types of learning"

print(f"Query: {query}")

print("\nA) LLM-based Document Summary Retriever:")
print("Uses LLM to select relevant documents based on summaries")
try:
    nodes_llm = doc_summary_retriever_llm.retrieve(query)
    print(f"Retrieved {len(nodes_llm)} nodes")
    for i, node in enumerate(nodes_llm[:2], 1):
        print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Document summary)")
        print(f"   Text: {node.text[:80]}...")
        print()
except Exception as e:
    print(f"LLM-based retrieval demo: {str(e)[:100]}...")

print("B) Embedding-based Document Summary Retriever:")
print("Uses vector similarity between query and document summaries")
try:
    nodes_emb = doc_summary_retriever_embedding.retrieve(query)
    print(f"Retrieved {len(nodes_emb)} nodes")
    for i, node in enumerate(nodes_emb[:2], 1):
        print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Document summary)")
        print(f"   Text: {node.text[:80]}...")
        print()
except Exception as e:
    print(f"Embedding-based retrieval demo: {str(e)[:100]}...")

print("Document Summary Index workflow:")
print("1. Generates summaries for each document using LLM")
print("2. Uses summaries to select relevant documents")
print("3. Returns full content from selected documents")

3. DOCUMENT SUMMARY INDEX RETRIEVERS
Query: different types of learning

A) LLM-based Document Summary Retriever:
Uses LLM to select relevant documents based on summaries
Retrieved 3 nodes
1. Score: 9.0000
   Text: Unsupervised learning finds hidden patterns in data without labeled examples....

2. Score: 8.0000
   Text: Reinforcement learning is a type of machine learning where agents learn to make ...

B) Embedding-based Document Summary Retriever:
Uses vector similarity between query and document summaries
Retrieved 3 nodes
1. (Document summary)
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to...

2. (Document summary)
   Text: Unsupervised learning finds hidden patterns in data without labeled examples....

Document Summary Index workflow:
1. Generates summaries for each document using LLM
2. Uses summaries to select relevant documents
3. Returns full content from selected documents


### Auto Merging Retriever (Hierarchical Context Preservation)

Auto Merging Retriever preserves context in long documents by indexing child chunks for precise search and returning parent chunks when enough related children match.

- **How it works**: Documents are split hierarchically into parent/child nodes, children are vector-indexed, parents are docstore-stored, and matching children trigger parent-level merging.

- **When to use**: Best for long structured documents (legal, research, technical) where small chunks lose surrounding context.

- **Config**: Set hierarchical chunk_sizes (e.g., [512, 256, 128]), chunk_overlap, and storage for both vector store and docstore.

- **Strengths**: Preserves broader context automatically while keeping fine-grained retrieval.

- **Limitations**: More setup complexity, higher storage overhead, and less value for short documents.

In [23]:
print("=" * 60)
print("4. AUTO MERGING RETRIEVER")
print("=" * 60)

# Create hierarchical nodes
node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[512, 256, 128]
)

hier_nodes = node_parser.get_nodes_from_documents(lab.documents)

# Create storage context with all nodes
from llama_index.core import StorageContext
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.vector_stores import SimpleVectorStore

docstore = SimpleDocumentStore()
docstore.add_documents(hier_nodes)

storage_context = StorageContext.from_defaults(docstore=docstore)

# Create base index
base_index = VectorStoreIndex(hier_nodes, storage_context=storage_context)
base_retriever = base_index.as_retriever(similarity_top_k=6)

# Create auto-merging retriever
auto_merging_retriever = AutoMergingRetriever(
    base_retriever, 
    storage_context,
    verbose=True
)

query = DEMO_QUERIES["advanced"]  # "How do neural networks work in deep learning?"
nodes = auto_merging_retriever.retrieve(query)

print(f"Query: {query}")
print(f"Auto-merged to {len(nodes)} nodes")
for i, node in enumerate(nodes[:3], 1):
    print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Auto-merged)")
    print(f"   Text: {node.text[:120]}...")
    print()

4. AUTO MERGING RETRIEVER
> Merging 1 nodes into parent node.
> Parent node id: 912c0b4a-3c2f-4a22-a521-6a1e436f2682.
> Parent node text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs.

Query: How do neural networks work in deep learning?
Auto-merged to 2 nodes
1. Score: 0.8570
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in data....

2. Score: 0.6956
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs....



### Recursive Retriever - Multi-Level Reference Following

Recursive Retriever follows links between nodes (such as citations and metadata references) to fetch related content across documents and levels.

- **How it works**: Traverses node references, supports chunk and metadata links, runs sub-queries on linked retrievers/query engines, and builds an interconnected retrieval graph.

- **Reference types**: Uses chunk references (child to parent context) and metadata references (for example summaries/questions/citations to source content).

- **When to use**: Best for citation-heavy papers, cross-referenced docs, connected knowledge bases, and nodes linked to structured data.

- **Configuration**: Define retriever_dict, query_engine_dict, and node metadata references.
Key capability: Retrieves related information across documents by following reference chains.

- **Strengths**: Handles complex relationships, enables multi-step retrieval, and improves coverage across linked sources.

- **Limitations**: Needs careful setup, can be costly on deep chains, is harder to debug, and can over-retrieve if poorly tuned.

*Based on: https://docs.llamaindex.ai/en/stable/examples/retrievers/recurisve_retriever_nodes_braintrust/*


In [24]:
print("=" * 60)
print("5. RECURSIVE RETRIEVER")
print("=" * 60)

# Create documents with references
docs_with_refs = []
for i, doc in enumerate(lab.documents):
    # Add reference metadata
    ref_doc = Document(
        text=doc.text,
        metadata={
            "doc_id": f"doc_{i}",
            "references": [f"doc_{j}" for j in range(len(lab.documents)) if j != i][:2]
        }
    )
    docs_with_refs.append(ref_doc)

# Create index with referenced documents
ref_index = VectorStoreIndex.from_documents(docs_with_refs)

# Create retriever mapping
retriever_dict = {
    f"doc_{i}": ref_index.as_retriever(similarity_top_k=1)
    for i in range(len(docs_with_refs))
}

# Base retriever
base_retriever = ref_index.as_retriever(similarity_top_k=2)

# Add the root retriever to the dictionary
retriever_dict["vector"] = base_retriever

# Recursive retriever
recursive_retriever = RecursiveRetriever(
    "vector",
    retriever_dict=retriever_dict,
    query_engine_dict={},
    verbose=True
)

query = DEMO_QUERIES["applications"]  # "What are the applications of AI?"
try:
    nodes = recursive_retriever.retrieve(query)
    print(f"Query: {query}")
    print(f"Recursively retrieved {len(nodes)} nodes")
    for i, node in enumerate(nodes[:3], 1):
        print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Recursive)")
        print(f"   Text: {node.text[:100]}...")
        print()
except Exception as e:
    print(f"Query: {query}")
    print(f"Recursive retriever demo: {str(e)}")
    print("Note: Recursive retriever requires specific node reference setup")
    
    # Fallback to basic retrieval for demonstration
    print("\nFalling back to basic retrieval demonstration...")
    base_nodes = base_retriever.retrieve(query)
    for i, node in enumerate(base_nodes[:2], 1):
        print(f"{i}. Score: {node.score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        print()

5. RECURSIVE RETRIEVER
Retrieving with query id None: What are the applications of AI?
Retrieving text node: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.
Retrieving text node: Natural language processing enables computers to understand, interpret, and generate human language.
Query: What are the applications of AI?
Recursively retrieved 2 nodes
1. Score: 0.6907
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Score: 0.6500
   Text: Natural language processing enables computers to understand, interpret, and generate human language....



### Query Fusion Retriever (Multi-Query + Multi-Retriever Fusion)

Query Fusion Retriever combines outputs from multiple retrievers and can generate query variants with an LLM, then fuses results to improve recall.

- **How it works**: Runs multiple retrievers and optional query rewrites, then merges outputs with a fusion strategy to reduce query-phrasing sensitivity.

- **Core capabilities**: Supports multi-retriever retrieval, optional LLM query variation, and strategy-based result fusion.

- **Fusion modes**: Uses Reciprocal Rank Fusion (reciprocal_rerank), Relative Score Fusion (relative_score), or Distribution-Based Fusion (dist_based_score).

- **When to use**: Best for ambiguous/complex queries, exploratory search, and cases needing both semantic and keyword signals.

- **Configuration**: Key knobs are num_queries, mode, similarity_top_k, and use_async.

- **Strengths**: Improves recall, handles query variation better, and combines complementary retriever strengths.

- **Limitations**: Higher compute/cost, added LLM cost for rewrites, potential noise if fusion is poorly tuned, and more setup complexity.

In [25]:
print("=" * 60)
print("6. QUERY FUSION RETRIEVER - OVERVIEW")
print("=" * 60)

# Create base retriever
base_retriever = lab.vector_index.as_retriever(similarity_top_k=3)

query = DEMO_QUERIES["comprehensive"]  # "What are the main approaches to machine learning?"
print(f"Query: {query}")
print("QueryFusionRetriever generates multiple query variations and fuses results")
print("using one of three sophisticated fusion modes.")

print("\nOverview of Fusion Modes:")
print("1. RECIPROCAL_RERANK: Uses reciprocal rank fusion (most robust)")
print("2. RELATIVE_SCORE: Preserves score magnitudes (most interpretable)")  
print("3. DIST_BASED_SCORE: Statistical normalization (most sophisticated)")

print("\nDemonstration workflow:")
print("Each subsection below explores one fusion mode in detail with:")
print("- Theoretical explanation of the fusion method")
print("- Live demonstration using QueryFusionRetriever")
print("- Manual implementation showing the underlying mathematics")
print("- Use case recommendations and trade-offs")

print(f"\nUsing consistent test query throughout: '{query}'")
print("This allows direct comparison of how each fusion mode handles the same input.")

print("\nProceed to subsections 6.1, 6.2, and 6.3 for detailed demonstrations...")

6. QUERY FUSION RETRIEVER - OVERVIEW
Query: What are the main approaches to machine learning?
QueryFusionRetriever generates multiple query variations and fuses results
using one of three sophisticated fusion modes.

Overview of Fusion Modes:
1. RECIPROCAL_RERANK: Uses reciprocal rank fusion (most robust)
2. RELATIVE_SCORE: Preserves score magnitudes (most interpretable)
3. DIST_BASED_SCORE: Statistical normalization (most sophisticated)

Demonstration workflow:
Each subsection below explores one fusion mode in detail with:
- Theoretical explanation of the fusion method
- Live demonstration using QueryFusionRetriever
- Manual implementation showing the underlying mathematics
- Use case recommendations and trade-offs

Using consistent test query throughout: 'What are the main approaches to machine learning?'
This allows direct comparison of how each fusion mode handles the same input.

Proceed to subsections 6.1, 6.2, and 6.3 for detailed demonstrations...


##### 6.1 Reciprocal Rank Fusion (RRF) Mode

RRF is a robust QueryFusionRetriever mode that merges ranked results from multiple query variants using reciprocal rank scores for stable fusion.

- **How it works:** For each query variant, it scores documents with `1 / (rank + k)` (typically `k=60`), sums scores across variants, and re-ranks by total.  

- **Formula:** `RRF_score(d) = Σ (1 / (rank_i(d) + k))`.  

- **Why it works:** It is scale-invariant, outlier-resistant, query-agnostic, and widely validated in IR practice.  

- **When to use:** Use as the default for most fusion tasks, especially when query variants have uneven quality and you need stable production behavior.  

- **Advantages:** Stable, simple, low-tuning, efficient, and tolerant of varying result-list lengths.  

- **Limitations:** Ignores absolute score magnitudes and gives equal weight to all query variants.

*Based on: https://docs.llamaindex.ai/en/stable/examples/retrievers/reciprocal_rerank_fusion/*


In [26]:
print("=" * 60)
print("6.1 RECIPROCAL RANK FUSION MODE DEMONSTRATION")
print("=" * 60)

# Create QueryFusionRetriever with RRF mode
base_retriever = lab.vector_index.as_retriever(similarity_top_k=5)

print("Testing QueryFusionRetriever with reciprocal_rerank mode:")
print("This demonstrates how RRF works within the query fusion framework")

# Use the same query for consistency across all fusion modes
query = DEMO_QUERIES["comprehensive"]  # "What are the main approaches to machine learning?"

try:
    # Create query fusion retriever with RRF mode
    rrf_query_fusion = QueryFusionRetriever(
        [base_retriever],
        similarity_top_k=3,
        num_queries=3,
        mode="reciprocal_rerank",
        use_async=False,
        verbose=True
    )
    
    print(f"\nQuery: {query}")
    print("QueryFusionRetriever will:")
    print("1. Generate query variations using LLM")
    print("2. Retrieve results for each variation")
    print("3. Apply Reciprocal Rank Fusion")
    
    nodes = rrf_query_fusion.retrieve(query)
    
    print(f"\nRRF Query Fusion Results:")
    for i, node in enumerate(nodes, 1):
        print(f"{i}. Final RRF Score: {node.score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        print()
    
    print("RRF Benefits in Query Fusion Context:")
    print("- Automatically handles query variations of different quality")
    print("- No bias toward queries that return higher raw scores")
    print("- Stable performance across diverse query formulations")
    
except Exception as e:
    print(f"QueryFusionRetriever error: {e}")
    print("Demonstrating RRF concept manually with query variations...")
    
    # Manual demonstration with query variations derived from the main query
    query_variations = [
        DEMO_QUERIES["comprehensive"],  # Original query
        "machine learning approaches and methods",
        "different ML techniques and algorithms"
    ]
    
    print("Manual RRF with Query Variations:")
    all_results = {}
    
    for i, query_var in enumerate(query_variations):
        print(f"\nQuery variation {i+1}: {query_var}")
        nodes = base_retriever.retrieve(query_var)
        
        # Apply RRF scoring
        for rank, node in enumerate(nodes):
            node_id = node.node.node_id
            if node_id not in all_results:
                all_results[node_id] = {
                    'node': node,
                    'rrf_score': 0,
                    'query_ranks': []
                }
            
            # Calculate RRF contribution: 1 / (rank + k)
            k = 60  # Standard RRF parameter
            rrf_contribution = 1.0 / (rank + 1 + k)
            all_results[node_id]['rrf_score'] += rrf_contribution
            all_results[node_id]['query_ranks'].append((i, rank + 1))
    
    # Sort by final RRF score
    sorted_results = sorted(
        all_results.values(), 
        key=lambda x: x['rrf_score'], 
        reverse=True
    )
    
    print(f"\nCombined RRF Results (top 3):")
    for i, result in enumerate(sorted_results[:3], 1):
        print(f"{i}. Final RRF Score: {result['rrf_score']:.4f}")
        print(f"   Query ranks: {result['query_ranks']}")
        print(f"   Text: {result['node'].text[:100]}...")
        print()
    
    print("RRF Formula Demonstration:")
    print("For each document: RRF_score = Σ(1 / (rank + 60))")
    print("- Rank 1 in query: 1/(1+60) = 0.0164")
    print("- Rank 2 in query: 1/(2+60) = 0.0161")
    print("- Rank 3 in query: 1/(3+60) = 0.0159")
    print("Documents appearing in multiple queries get higher combined scores")

6.1 RECIPROCAL RANK FUSION MODE DEMONSTRATION
Testing QueryFusionRetriever with reciprocal_rerank mode:
This demonstrates how RRF works within the query fusion framework

Query: What are the main approaches to machine learning?
QueryFusionRetriever will:
1. Generate query variations using LLM
2. Retrieve results for each variation
3. Apply Reciprocal Rank Fusion
Generated queries:
1. "What are the primary methods in machine learning?"
2. "How many main techniques exist in machine learning?"

RRF Query Fusion Results:
1. Final RRF Score: 0.0500
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Final RRF Score: 0.0489
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...

3. Final RRF Score: 0.0487
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs....

RRF Benefits in Query Fusion Context:
- Automatically handles que

##### 6.2 Relative Score Fusion Mode

Relative Score Fusion normalizes each query variant’s retrieval scores by that query’s max score, then combines them to preserve confidence magnitude across variants.

- **How it works**: For each query variation, compute normalized_score = score / max_score, then merge with weighted sum/average.

- **Why useful**: It preserves confidence strength, avoids scale dominance, and yields interpretable combined scores.

- **When to use**: Best when retriever/embedding scores are trustworthy and you want proportional contribution by confidence.

- **Config behavior**: QueryFusionRetriever handles normalization automatically and uses equal query weights by default.

- **Advantages**: Keeps magnitude information, is intuitive, and is more interpretable than rank-only fusion.

- **Limitations**: Sensitive to outliers and weak when underlying scores are noisy or poorly calibrated.

In [27]:
print("=" * 60)
print("6.2 RELATIVE SCORE FUSION MODE DEMONSTRATION")
print("=" * 60)

base_retriever = lab.vector_index.as_retriever(similarity_top_k=5)

print("Testing QueryFusionRetriever with relative_score mode:")
print("This mode preserves score magnitudes while normalizing across query variations")

# Use the same query for consistency across all fusion modes
query = DEMO_QUERIES["comprehensive"]  # "What are the main approaches to machine learning?"

try:
    # Create query fusion retriever with relative score mode
    rel_score_fusion = QueryFusionRetriever(
        [base_retriever],
        similarity_top_k=3,
        num_queries=3,
        mode="relative_score",
        use_async=False,
        verbose=False
    )
    
    print(f"\nQuery: {query}")
    print("QueryFusionRetriever with relative_score will:")
    print("1. Generate query variations")
    print("2. Normalize scores within each variation (score/max_score)")
    print("3. Combine normalized scores")
    
    nodes = rel_score_fusion.retrieve(query)
    
    print(f"\nRelative Score Fusion Results:")
    for i, node in enumerate(nodes, 1):
        print(f"{i}. Combined Relative Score: {node.score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        print()
    
    print("Relative Score Benefits in Query Fusion:")
    print("- Preserves confidence information from embedding model")
    print("- Ensures fair contribution from each query variation")
    print("- More interpretable than rank-only methods")
    
except Exception as e:
    print(f"QueryFusionRetriever error: {e}")
    print("Demonstrating Relative Score concept manually...")
    
    # Manual demonstration with query variations derived from the main query
    query_variations = [
        DEMO_QUERIES["comprehensive"],  # Original query
        "machine learning approaches and methods",
        "different ML techniques and algorithms"
    ]
    
    print("Manual Relative Score Fusion with Query Variations:")
    all_results = {}
    query_max_scores = []
    
    # Step 1: Get results and find max scores for each query
    for i, query_var in enumerate(query_variations):
        print(f"\nQuery variation {i+1}: {query_var}")
        nodes = base_retriever.retrieve(query_var)
        scores = [node.score or 0 for node in nodes]
        max_score = max(scores) if scores else 1.0
        query_max_scores.append(max_score)
        
        print(f"Max score for this query: {max_score:.4f}")
        
        # Store results with normalization info
        for node in nodes:
            node_id = node.node.node_id
            original_score = node.score or 0
            normalized_score = original_score / max_score if max_score > 0 else 0
            
            if node_id not in all_results:
                all_results[node_id] = {
                    'node': node,
                    'combined_score': 0,
                    'contributions': []
                }
            
            all_results[node_id]['combined_score'] += normalized_score
            all_results[node_id]['contributions'].append({
                'query': i,
                'original': original_score,
                'normalized': normalized_score
            })
    
    # Step 2: Sort by combined relative score
    sorted_results = sorted(
        all_results.values(),
        key=lambda x: x['combined_score'],
        reverse=True
    )
    
    print(f"\nCombined Relative Score Results (top 3):")
    for i, result in enumerate(sorted_results[:3], 1):
        print(f"{i}. Combined Score: {result['combined_score']:.4f}")
        print(f"   Score breakdown:")
        for contrib in result['contributions']:
            print(f"     Query {contrib['query']}: {contrib['original']:.3f} → {contrib['normalized']:.3f}")
        print(f"   Text: {result['node'].text[:100]}...")
        print()
    
    print("Relative Score Normalization Process:")
    print("1. For each query variation, find max_score")
    print("2. Normalize: normalized_score = original_score / max_score")
    print("3. Sum normalized scores across all query variations")
    print("4. Documents with consistently high scores across queries win")

6.2 RELATIVE SCORE FUSION MODE DEMONSTRATION
Testing QueryFusionRetriever with relative_score mode:
This mode preserves score magnitudes while normalizing across query variations

Query: What are the main approaches to machine learning?
QueryFusionRetriever with relative_score will:
1. Generate query variations
2. Normalize scores within each variation (score/max_score)
3. Combine normalized scores
QueryFusionRetriever error: (ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: a9cc383f-8e9f-444a-9a45-dacb26e844f0)')
Demonstrating Relative Score concept manually...
Manual Relative Score Fusion with Query Variations:

Query variation 1: What are the main approaches to machine learning?
Max score for this query: 0.8070

Query variation 2: machine learning approaches and methods
Max score for this query: 0.8274

Query variation 3: different ML techniques and algorithms
Max score for this query: 0.7526

Combined Relative Score Results (

##### 6.3 Distribution-Based Score Fusion Mode

Distribution-Based Score Fusion normalizes each query variant’s scores using distribution statistics, then combines them for robust fusion under variable score quality.

- **How it works**: It analyzes per-query score distributions, applies statistical normalization (for example z-score or percentile), and merges results with confidence-aware weighting.

- **Why it helps**: It is robust to scale differences, outliers, and uneven reliability across query variants.

- **When to use**: Best for complex/noisy retrieval where query variants produce very different score distributions.

- **Advanced behavior**: Supports automatic distribution analysis, confidence weighting, outlier handling, and variable result-size robustness.

- **Advantages**: Most statistically principled and adaptive fusion approach for heterogeneous score behavior.

- **Limitations**: Most compute-heavy, needs enough results for stable estimates, and is less interpretable than simpler fusion modes.

In [28]:
print("=" * 60)
print("6.3 DISTRIBUTION-BASED SCORE FUSION MODE DEMONSTRATION")
print("=" * 60)

base_retriever = lab.vector_index.as_retriever(similarity_top_k=8)

print("Testing QueryFusionRetriever with dist_based_score mode:")
print("This mode uses statistical analysis for the most sophisticated score fusion")

# Use the same query for consistency across all fusion modes
query = DEMO_QUERIES["comprehensive"]  # "What are the main approaches to machine learning?"

try:
    # Create query fusion retriever with distribution-based mode
    dist_fusion = QueryFusionRetriever(
        [base_retriever],
        similarity_top_k=3,
        num_queries=3,
        mode="dist_based_score",
        use_async=False,
        verbose=False
    )
    
    print(f"\nQuery: {query}")
    print("QueryFusionRetriever with dist_based_score will:")
    print("1. Generate query variations")
    print("2. Analyze score distributions for each variation")
    print("3. Apply statistical normalization (z-score, percentiles)")
    print("4. Combine with distribution-aware weighting")
    
    nodes = dist_fusion.retrieve(query)
    
    print(f"\nDistribution-Based Fusion Results:")
    for i, node in enumerate(nodes, 1):
        print(f"{i}. Statistically Normalized Score: {node.score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        print()
    
    print("Distribution-Based Benefits in Query Fusion:")
    print("- Accounts for score distribution differences between query variations")
    print("- Statistically robust against outliers and noise")
    print("- Adapts weighting based on query variation reliability")
    
except Exception as e:
    print(f"QueryFusionRetriever error: {e}")
    print("Demonstrating Distribution-Based concept manually...")
    
    if not SCIPY_AVAILABLE:
        print("⚠️ Full statistical analysis requires scipy")
    
    # Manual demonstration with query variations derived from the main query
    query_variations = [
        DEMO_QUERIES["comprehensive"],  # Original query
        "machine learning approaches and methods",
        "different ML techniques and algorithms"
    ]
    
    print("Manual Distribution-Based Fusion with Query Variations:")
    all_results = {}
    variation_stats = []
    
    # Step 1: Collect results and analyze distributions
    for i, query_var in enumerate(query_variations):
        print(f"\nQuery variation {i+1}: {query_var}")
        nodes = base_retriever.retrieve(query_var)
        scores = [node.score or 0 for node in nodes]
        
        # Calculate distribution statistics
        mean_score = np.mean(scores) if scores else 0
        std_score = np.std(scores) if len(scores) > 1 else 1
        min_score = np.min(scores) if scores else 0
        max_score = np.max(scores) if scores else 1
        
        stats_info = {
            'mean': mean_score,
            'std': std_score,
            'min': min_score,
            'max': max_score,
            'nodes': nodes,
            'scores': scores
        }
        variation_stats.append(stats_info)
        
        print(f"Distribution stats: mean={mean_score:.3f}, std={std_score:.3f}")
        print(f"Score range: [{min_score:.3f}, {max_score:.3f}]")
        
        # Apply z-score normalization
        for node, score in zip(nodes, scores):
            node_id = node.node.node_id
            
            # Z-score normalization
            if std_score > 0:
                z_score = (score - mean_score) / std_score
            else:
                z_score = 0
            
            # Convert to [0,1] using sigmoid
            normalized_score = 1 / (1 + np.exp(-z_score))
            
            if node_id not in all_results:
                all_results[node_id] = {
                    'node': node,
                    'combined_score': 0,
                    'contributions': []
                }
            
            all_results[node_id]['combined_score'] += normalized_score
            all_results[node_id]['contributions'].append({
                'query': i,
                'original': score,
                'z_score': z_score,
                'normalized': normalized_score
            })
    
    # Step 2: Sort by combined distribution-based score
    sorted_results = sorted(
        all_results.values(),
        key=lambda x: x['combined_score'],
        reverse=True
    )
    
    print(f"\nCombined Distribution-Based Results (top 3):")
    for i, result in enumerate(sorted_results[:3], 1):
        print(f"{i}. Combined Score: {result['combined_score']:.4f}")
        print(f"   Statistical breakdown:")
        for contrib in result['contributions']:
            print(f"     Query {contrib['query']}: {contrib['original']:.3f} → "
                  f"z={contrib['z_score']:.2f} → {contrib['normalized']:.3f}")
        print(f"   Text: {result['node'].text[:100]}...")
        print()
    
    print("Distribution-Based Process:")
    print("1. Calculate mean and std for each query variation")
    print("2. Z-score normalize: z = (score - mean) / std")
    print("3. Sigmoid transform: normalized = 1 / (1 + exp(-z))")
    print("4. Sum normalized scores across variations")
    print("5. Results reflect statistical significance across all query forms")

# Show fusion mode comparison summary
print("\n" + "=" * 60)
print("FUSION MODES COMPARISON SUMMARY")
print("=" * 60)
print("All three modes tested with the same query for direct comparison:")
print(f"Query: {query}")
print()
print("Mode Characteristics:")
print("• RRF (reciprocal_rerank): Most robust, rank-based, scale-invariant")
print("• Relative Score: Preserves confidence, normalizes by max score")  
print("• Distribution-Based: Most sophisticated, statistical normalization")
print()
print("Choose based on your use case:")
print("- Production stability → RRF")
print("- Score interpretability → Relative Score")
print("- Statistical robustness → Distribution-Based")

6.3 DISTRIBUTION-BASED SCORE FUSION MODE DEMONSTRATION
Testing QueryFusionRetriever with dist_based_score mode:
This mode uses statistical analysis for the most sophisticated score fusion

Query: What are the main approaches to machine learning?
QueryFusionRetriever with dist_based_score will:
1. Generate query variations
2. Analyze score distributions for each variation
3. Apply statistical normalization (z-score, percentiles)
4. Combine with distribution-aware weighting

Distribution-Based Fusion Results:
1. Statistically Normalized Score: 0.8759
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Statistically Normalized Score: 0.5705
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs....

3. Statistically Normalized Score: 0.5623
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...

Distribution-Based Benefits in

### 📝 Exercise 1 - Build a Custom Hybrid Retriever

Your task is to create a hybrid retriever that combines both vector similarity and BM25 keyword search for improved results.

**Requirements:**
- Use both Vector Index Retriever and BM25 Retriever
- Implement a simple score fusion mechanism which takes a weighted average of normalized scores
- Test with different query types (semantic vs keyword-focused)

**Important Note**: Node IDs from different retrievers won't match even for the same content, so we need to match by text content instead.

```python
# TODO: Implement hybrid retriever
# Step 1: Create both retrievers
vector_retriever = # Your code here
bm25_retriever = # Your code here

# Step 2: Implement score fusion
def hybrid_retrieve(query, top_k=5):
    # Your implementation here
    pass

# Step 3: Test with different queries
test_queries = [
    "What is machine learning?",  # Semantic query
    "neural networks deep learning",  # Keyword query
    "supervised learning techniques"  # Mixed query
]
```

In [32]:
### Put your solution here ###

# TODO: Implement hybrid retriever
# Step 1: Create both retrievers
vector_retriever = VectorIndexRetriever(
    index=lab.vector_index,
) # Your code here
bm25_retriever = BM25Retriever.from_defaults(
    nodes=lab.nodes,
    stemmer=Stemmer.Stemmer("english"),
    language="english"
)
# Your code here

# Step 2: Implement score fusion
def hybrid_retrieve(query, top_k=5):
    # Your implementation here
    dist_fusion = QueryFusionRetriever(
        [vector_retriever, bm25_retriever],
        similarity_top_k=top_k,
        num_queries=3,
        mode="dist_based_score",
        use_async=False,
        verbose=False
    )

    nodes = dist_fusion.retrieve(query)

    for i, node in enumerate(nodes, 1):
        print(f"{i}. Statistically Normalized Score: {node.score:.4f}")
        print(f"   Text: {node.text[:100]}...")
        print()

# Step 3: Test with different queries
test_queries = [
    "What is machine learning?",  # Semantic query
    "neural networks deep learning",  # Keyword query
    "supervised learning techniques"  # Mixed query
]

for query in test_queries:
    print("=" * 15)
    print(query)
    print("-" * 15)
    hybrid_retrieve(query)

What is machine learning?
---------------
1. Statistically Normalized Score: 0.5000
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Statistically Normalized Score: 0.3889
   Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re...

3. Statistically Normalized Score: 0.0556
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...

4. Statistically Normalized Score: 0.0556
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs....

neural networks deep learning
---------------
1. Statistically Normalized Score: 0.6667
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...

2. Statistically Normalized Score: 0.1667
   Text: Unsupervised learning finds hidden patterns in data without labeled examples....

3. Statist

### 📝 Exercise 2 - Create a Production RAG Pipeline

Build a complete RAG pipeline that uses multiple retrieval strategies and includes evaluation metrics.

**Requirements:**
- Implement retrieval with multiple strategies
- Add query routing logic
- Include basic evaluation metrics that evaluate whether the pipeline succeeded or failed
- Handle edge cases and errors

```python
# TODO: Implement production RAG pipeline
class ProductionRAGPipeline:
    def __init__(self, index, llm):
        self.index = index
        self.llm = llm
        # Your initialization code here
    
    def query(self, question, strategy="auto"):
        # Your implementation here
        pass
    
    def evaluate(self, test_queries, expected_answers):
        # Your evaluation implementation here
        pass

# Test the pipeline
pipeline = ProductionRAGPipeline(lab.vector_index, llm)
```


In [33]:
### Put your solution here ###
# TODO: Implement production RAG pipeline
class ProductionRAGPipeline:
    def __init__(self, index, llm):
        self.index = index
        self.llm = llm
        self.vector_retriever = index.as_retriever(similarity_top_k=5)
        
    def _route_query(self, question):
        """Simple query routing based on question characteristics"""
        if any(word in question.lower() for word in ["what", "explain", "describe"]):
            return "semantic"
        elif any(word in question.lower() for word in ["list", "types", "examples"]):
            return "comprehensive"
        else:
            return "semantic"
    
    def query(self, question, strategy="auto"):
        try:
            # Route query if strategy is auto
            if strategy == "auto":
                strategy = self._route_query(question)
            
            # Retrieve relevant documents
            if strategy == "semantic":
                retriever = self.vector_retriever
                top_k = 3
            elif strategy == "comprehensive":
                retriever = self.vector_retriever
                top_k = 5
            else:
                retriever = self.vector_retriever
                top_k = 3
            
            # Get relevant documents
            relevant_docs = retriever.retrieve(question)
            
            # Prepare context
            context = "\n\n".join([doc.text for doc in relevant_docs[:top_k]])
            
            # Generate response
            prompt = f"""Based on the following context, please answer the question:

Context:
{context}

Question: {question}

Answer:"""
            
            try:
                response = self.llm.complete(prompt)
                return {
                    "answer": response.text,
                    "strategy": strategy,
                    "num_docs": len(relevant_docs),
                    "status": "success"
                }
            except Exception as e:
                return {
                    "answer": f"Based on the retrieved documents: {context[:200]}...",
                    "strategy": strategy,
                    "num_docs": len(relevant_docs),
                    "status": f"llm_error: {str(e)}"
                }
                
        except Exception as e:
            return {
                "answer": "I encountered an error processing your question.",
                "strategy": strategy,
                "num_docs": 0,
                "status": f"error: {str(e)}"
            }
    
    def evaluate(self, test_queries):
        results = []
        for query in test_queries:
            result = self.query(query)
            results.append({
                "query": query,
                "result": result,
                "success": result["status"] == "success"
            })
        
        success_rate = sum(1 for r in results if r["success"]) / len(results)
        return {
            "success_rate": success_rate,
            "results": results
        }

# Test the pipeline
pipeline = ProductionRAGPipeline(lab.vector_index, llm)

test_queries = [
    "What is machine learning?",
    "List different types of learning algorithms",
    "Explain neural networks"
]

print("Testing Production RAG Pipeline:")
for query in test_queries:
    result = pipeline.query(query)
    print(f"\nQuery: {query}")
    print(f"Strategy: {result['strategy']}")
    print(f"Status: {result['status']}")
    print(f"Answer: {result['answer'][:100]}...")

# Evaluate performance
evaluation = pipeline.evaluate(test_queries)
print(f"\nPipeline Success Rate: {evaluation['success_rate']:.2%}")

Testing Production RAG Pipeline:

Query: What is machine learning?
Strategy: semantic
Status: success
Answer:  Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn f...

Query: List different types of learning algorithms
Strategy: comprehensive
Status: success
Answer:  Based on the provided context, here are the different types of learning algorithms mentioned:
1. **...

Query: Explain neural networks
Strategy: semantic
Status: success
Answer:  A neural network is a type of machine learning model inspired by the structure and function of huma...

Pipeline Success Rate: 100.00%
